In [2]:
import csv
import random

# Load benign data from CSV file
def load_data(input_file):
    with open(input_file, 'r') as file:
        reader = csv.reader(file)
        header = next(reader)
        data = [row for row in reader]
    return header, data

def apply_attack(row, scenario):
    altered_row = row.copy()

    # Get necessary attributes
    voltage = float(altered_row[1])
    current = float(altered_row[2])
    power = float(altered_row[6])

    if scenario == "baseline":
        voltage_multiplier = random.uniform(0.95, 1.05)
        current_multiplier = random.uniform(0.95, 1.05)
        power_multiplier = random.uniform(0.95, 1.05)
    elif scenario == "weakload":
        voltage_multiplier = random.uniform(0.9, 0.95)
        current_multiplier = random.uniform(0.9, 0.95)
        power_multiplier = random.uniform(0.9, 0.95)
    elif scenario == "peakhour":
        voltage_multiplier = random.uniform(1.05, 1.1)
        current_multiplier = random.uniform(1.05, 1.1)
        power_multiplier = random.uniform(1.05, 1.1)
    elif scenario == "midnight":
        midnight_start = 22  # 22:00 (10 PM)
        midnight_end = 2     # 2:00 (2 AM)
        hour = int(row[0]) % 24
        if midnight_start <= hour or hour <= midnight_end:
            power_multiplier = random.uniform(1.1, 1.3)
        else:
            power_multiplier = random.uniform(0.95, 1.05)
        voltage_multiplier = random.uniform(0.95, 1.05)
        current_multiplier = random.uniform(0.95, 1.05)
    elif scenario == "evil-twin":
        voltage_multiplier = 1
        current_multiplier = 1
        power_multiplier = 2

    # Apply multipliers
    altered_voltage = voltage * voltage_multiplier
    altered_current = current * current_multiplier
    altered_power = power * power_multiplier

    altered_row[1] = str(altered_voltage)
    altered_row[2] = str(altered_current)
    altered_row[6] = str(altered_power)

    return altered_row



def generate_attack_data(benign_data, scenario):
    max_attack_data_points = 200
    num_attack_data_points = random.randint(1, max_attack_data_points)

    # Randomly select a subset of benign data points
    sampled_benign_data = random.sample(benign_data, num_attack_data_points)

    # Apply attack to the selected benign data points
    attack_data = [apply_attack(row, scenario) for row in sampled_benign_data]
    return attack_data


def save_attack_data(attack_data, output_file, label):
    with open(output_file, 'a', newline='') as file:
        writer = csv.writer(file)
        
        attack_data_with_label = [row + [label] for row in attack_data]
        writer.writerows(attack_data_with_label)


def main():
    benign_data_file = '/content/drive/MyDrive/abraham/Electricity_CWE.csv'
    header, benign_data = load_data(benign_data_file)

    attack_scenarios = ['baseline', 'weakload', 'peakhour', 'midnight', 'evil-twin']

    # Initialize all_data list with header
    all_data = [header + ['label']]

    # Save header to the output file
    all_data_file = '/content/drive/MyDrive/abraham/Electricity_CWE_Attack.csv'
    with open(all_data_file, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(all_data[0])

    for scenario in attack_scenarios:
        attack_data = generate_attack_data(benign_data, scenario)
        attack_data_with_label = [row + [scenario] for row in attack_data]
        all_data.extend(attack_data_with_label)

    # Save all data to a single CSV file
    with open(all_data_file, 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(all_data)

if __name__ == '_main_':
     main()
